# Import

In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import warnings
import matplotlib.pyplot as plt
import os 
import pandas as pd
import anndata as ad
import numpy as np
from pandas.api.types import CategoricalDtype
from hiara.src.utils.util import retrieve_net
from hiara.src.pathway_analysis.util import get_genesets, pathway_kde_func, get_hallmark, gsea_func, wrapper_gsea
from hiara.src.pathway_analysis.plots import plot_pathway_kde


# - plot
from hiara.src.feature_association.plots import heamap_plot_minor_cell_types

# plt.rcParams["figure.figsize"]=4,4
plt.rcParams["figure.dpi"]=150
# plt.rcParams["savefig.dpi"]=300
plt.rcParams["font.family"] = "Arial"

import warnings
warnings.filterwarnings("ignore")

# - dirs 
from hiara import PLOTS_DIR, OUTPUT_DIR, AGING_COHORTS ,\
     palette_datasets_pretty, palette_datasets, surrogate_names, \
     palette_treatment, colors_blind, palette_trend, palette_trend_2, palette_cell_types, palette_disease_effect, DATA_DIR, PRIOR_DIR
from hiara import retrieve_sig_stats, retrieve_adata
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
task_grn_inference_dir = '../../task_grn_inference'

# Population Aging

## TF activity

In [2]:
!cd ../ && python src/feature_association/run_analysis.py --feature-type tf_activity --data-type bulk --analysis-mode multi-cohort  --association-type continous
!cd ../ && python src/experiments/post_aging_analysis.py --feature-type tf_activity


MULTI-COHORT AGING ANALYSIS
Feature: tf_activity
Datasets: onek1k, abf300, aida, perez_sle
Cell types: CD4T, CD8T, NK, B, MONO
Data type: bulk
Promotor-based only: False


[1/3] Calculating features...
Loading data...
Calculating TF activity...
  - Promotor-based only: False
onek1k bulk
cell types:   0%|                                         | 0/5 [00:00<?, ?it/s]Retrieving consensus GRN for CD4T with min degree 2
... storing 'dataset' as categorical
... storing 'condition' as categorical
cell types:  20%|██████▌                          | 1/5 [00:04<00:19,  4.98s/it]Retrieving consensus GRN for CD8T with min degree 2
... storing 'dataset' as categorical
... storing 'condition' as categorical
cell types:  40%|█████████████▏                   | 2/5 [00:11<00:17,  5.97s/it]Retrieving consensus GRN for NK with min degree 2
... storing 'dataset' as categorical
... storing 'condition' as categorical
cell types:  60%|███████████████████▊             | 3/5 [00:14<00:09,  4.62s/it]Retrievin

In [ ]:
from hiara.src.feature_association.plots import plot_features_vs_datasets
plot_features_vs_datasets(cell_type='CD8T', data_type='bulk', datasets=AGING_COHORTS, features=None, 
                                    feature_type='tf_activity', sizes=(90, 100), min_degree=1, race='both', 
                                    filter_meta_significant=True,
                                    )

## Aging hallmarks

In [1]:
# !cd ../ && python src/feature_association/run_analysis.py  --feature-type aging_hallmarks --data-type bulk --analysis-mode multi-cohort --skip-features
# !cd ../ && python src/experiments/post_aging_analysis.py --feature-type aging_hallmarks

# Disease: SLE

In [ ]:
if True: #tf_activity
    !cd ../ && python src/feature_association/run_analysis.py  --datasets "perez_sle" --cell-types CD4T CD8T --data-type bulk --feature-type tf_activity --association-type grouped 
    !cd ../ && python src/experiments/post_condition_analysis.py --dataset "perez_sle" --analysis-type disease --data-type bulk --feature-type tf_activity --cell-types CD8T CD4T --skip-pathway


Analysis type: grouped
Dataset: perez_sle
Feature: tf_activity
Cell types: CD4T, CD8T


[1/3] Calculating features...
Loading data...
Calculating TF activity...
  - Promotor-based only: False
perez_sle bulk
cell types:   0%|                                         | 0/2 [00:00<?, ?it/s]Retrieving consensus GRN for CD4T with min degree 2
... storing 'dataset' as categorical
cell types:  50%|████████████████▌                | 1/2 [00:04<00:04,  4.57s/it]Retrieving consensus GRN for CD8T with min degree 2
... storing 'dataset' as categorical
cell types: 100%|█████████████████████████████████| 2/2 [00:11<00:00,  5.76s/it]
✓ Features calculated

[2/3] Computing condition statistics...
Association tf_activity with condition...
cell types: 100%|█████████████████████████████████| 2/2 [00:01<00:00,  1.68it/s]
['CD4T' 'CD8T']
✓ Condition stats saved: /vol/projects/jnourisa//output//features//tf_activity/stats/stats_perez_sle_bulk.csv

Significant features (FDR < 0.05) in meta-analysis:
cell_typ

In [75]:
if True: #aging_hallmarks
    # !cd ../ && bash scripts/experiment/run_condition_analysis.sh --dataset "SLE_European" --cell-types "CD4T CD8T" --data-type bulk --feature-type aging_hallmarks 
    !cd ../ && python src/experiments/post_condition_analysis.py --dataset "SLE_European" --analysis-type disease --data-type bulk --feature-type aging_hallmarks --cell-types CD8T CD4T --skip-pathway

Unified Condition Analysis - Disease
Dataset: SLE_European
Analysis type: disease
Data type: bulk
Feature type: aging_hallmarks
Output directory: /Users/jno24/Documents/projs/ongoing/ciim/base_folder/output//plots/
Cell types: CD8T, CD4T
Significance threshold: 0.05
Total significant features per cell type:
cell_type
CD4T    214
CD8T    150
Name: gene, dtype: int64
Generating aging overlap plot...
  Processing cell type: CD8T
    Overlap with aging genes: 111
      Same direction: 108 (97.3%)
      Opposite direction: 1 (0.9%)
    Saved: /Users/jno24/Documents/projs/ongoing/ciim/base_folder/output//plots/condition_aging_overlap_SLE_European_CD8T.png
  Processing cell type: CD4T
    Overlap with aging genes: 64
      Same direction: 57 (89.1%)
      Opposite direction: 7 (10.9%)
    Saved: /Users/jno24/Documents/projs/ongoing/ciim/base_folder/output//plots/condition_aging_overlap_SLE_European_CD4T.png


# Perturbations

In [3]:
# !cd ../ && bash src/feature_association/run_analysis.py --dataset op --cell-types "CD4T" --data-type bulk --feature-type tf_activity --association-type grouped 
# !cd ../ && python src/experiments/post_condition_analysis.py --dataset op --analysis-type perturbation --data-type bulk --feature-type tf_activity --cell-types CD4T --agreement opposite

!cd ../ && python src/feature_association/run_analysis.py --dataset CXCL9 --cell-types CD4T --data-type sc --feature-type tf_activity --association-type grouped 
!cd ../ && python src/experiments/post_condition_analysis.py --dataset CXCL9 --analysis-type perturbation --data-type sc --feature-type tf_activity --cell-types CD4T --agreement opposite --skip-overview --skip-pathway

# !cd ../ && bash src/feature_association/run_analysis.py --dataset parsebioscience --cell-types "CD4T CD8T" --data-type bulk --feature-type tf_activity --analysis_type condition
# !cd ../ && python src/experiments/post_condition_analysis.py --dataset parsebioscience --analysis-type perturbation --data-type bulk --feature-type tf_activity --cell-types CD4T CD8T --agreement opposite


Analysis type: grouped
Dataset: CXCL9
Feature: tf_activity
Cell types: CD4T


[1/3] Calculating features...
Loading data...
Calculating TF activity...
  - Promotor-based only: False
CXCL9 sc
cell types:   0%|                                         | 0/1 [00:00<?, ?it/s]Retrieving consensus GRN for CD4T with min degree 2
... storing 'dataset' as categorical
... storing 'donor_age' as categorical
cell types: 100%|█████████████████████████████████| 1/1 [00:13<00:00, 13.52s/it]
✓ Features calculated

[2/3] Computing condition statistics...
Association tf_activity with condition...
cell types: 100%|█████████████████████████████████| 1/1 [00:50<00:00, 50.34s/it]
['CD4T']
✓ Condition stats saved: /home/jnourisa/projs/ongoing/hiara//output//features//tf_activity/stats/stats_CXCL9_sc.csv

Significant features (FDR < 0.05) in meta-analysis:
cell_type  condition              
CD4T       24 h LPS + ruxolitinib     185
           24 h RPMI + ruxolitinib    166
Name: gene, dtype: int64
Unified Con

In [5]:
# !cd ../ && bash scripts/experiment/run_condition_analysis.sh --dataset op --cell-types "CD4T" --data-type sc --feature-type aging_hallmarks 
!cd ../ && python src/experiments/post_condition_analysis.py --dataset op --analysis-type perturbation --data-type sc --feature-type aging_hallmarks --cell-types CD4T --agreement opposite --skip-overview --skip-pathway

Unified Condition Analysis - Perturbation
Dataset: op
Analysis type: perturbation
Data type: sc
Feature type: aging_hallmarks
Output directory: /vol/projects/jnourisa//output//plots/
Cell types: CD4T
Significance threshold: 0.05
Total significant features per cell type:
cell_type
CD4T    98
CD8T     0
NK       0
B        0
MONO     0
Name: gene, dtype: int64
Generating aging overlap plot...
Traceback (most recent call last):
  File "/home/jnourisa/projs/ongoing/ciim/src/experiments/post_condition_analysis.py", line 1418, in <module>
    main()
  File "/home/jnourisa/projs/ongoing/ciim/src/experiments/post_condition_analysis.py", line 1410, in main
    plot_aging_overlap(
  File "/home/jnourisa/projs/ongoing/ciim/src/experiments/post_condition_analysis.py", line 247, in plot_aging_overlap
    aging_stats_sig = retrieve_sig_stats(type='bulk', feature_type=feature_type).drop_duplicates(subset=['cell_type', 'gene'])
  File "/home/jnourisa/projs/ongoing/ciim/src/feature_association/helper.p

In [ ]:
!cd ../ && bash scripts/experiment/run_condition_analysis.sh --dataset soundlife --cell-types "CD4T CD8T" --data-type metacell --feature-type tf_activity
# !cd ../ && python src/experiments/post_condition_analysis.py --dataset soundlife --analysis-type perturbation --data-type bulk --feature-type tf_activity --cell-types CD4T CD8T

Failed to connect to bus: No such file or directory


/home/jnourisa/.bashrc: line 60: cd: projs/ongoing/: No such file or directory
Condition Analysis (Unified)

Configuration:
  Dataset: soundlife
  Cell types: CD4T CD8T
  Feature type: tf_activity
  Data type: metacell
  Skip feature calculation: false


Running analysis...

usage: run_analysis.py [-h] --dataset DATASET
                       [--cell-types CELL_TYPES [CELL_TYPES ...]]
                       [--feature-type {tf_activity,gene_expression}]
                       [--data-type {bulk,sc}] [--skip-features]
                       [--association-type {spearman,pearson}]
run_analysis.py: error: argument --data-type: invalid choice: 'metacell' (choose from 'bulk', 'sc')


# Soundlife cohort

In [2]:
!cd ../ && python src/feature_association/run_analysis.py --analysis-mode single-cohort --datasets soundlife --cell-types CD8T CD4T MONO NK --data-type bulk --feature-type tf_activity --association-type continous 
!cd ../ && python src/experiments/post_condition_analysis.py --dataset soundlife --analysis-type aging --data-type bulk --feature-type tf_activity --cell-types CD8T CD4T MONO NK  --skip-overview --skip-pathway


Analysis type: continous
Dataset: soundlife
Feature: tf_activity
Cell types: CD8T, CD4T, MONO, NK


[1/3] Calculating features...
Loading data...
Calculating TF activity...
  - Promotor-based only: False
soundlife bulk
cell types:   0%|                                         | 0/4 [00:00<?, ?it/s]Retrieving consensus GRN for CD8T with min degree 2
... storing 'dataset' as categorical
... storing 'donor_age' as categorical
... storing 'condition' as categorical
cell types:  25%|████████▎                        | 1/4 [00:08<00:26,  8.73s/it]Retrieving consensus GRN for CD4T with min degree 2
... storing 'dataset' as categorical
... storing 'donor_age' as categorical
... storing 'condition' as categorical
cell types:  50%|████████████████▌                | 2/4 [00:12<00:12,  6.00s/it]Retrieving consensus GRN for MONO with min degree 2
... storing 'dataset' as categorical
... storing 'donor_age' as categorical
... storing 'condition' as categorical
cell types:  75%|██████████████████████